In [17]:
import pandas as pd
import numpy as np
from geopy.distance import distance
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# Load dataset
data = pd.read_csv('uber.csv')

# Drop unnecessary columns
data = data.drop(columns=['Unnamed: 0', 'key'])

# Check for null values and drop rows with any null values
data = data.dropna()

# Drop duplicate rows, if any
data = data.drop_duplicates()


# Filter out rows with invalid latitude and longitude values
data = data[(data.pickup_latitude < 90) & (data.dropoff_latitude < 90) &
            (data.pickup_latitude > -90) & (data.dropoff_latitude > -90) &
            (data.pickup_longitude < 180) & (data.dropoff_longitude < 180) &
            (data.pickup_longitude > -180) & (data.dropoff_longitude > -180)]


# Convert 'pickup_datetime' to datetime format and extract time-based features
data['pickup_datetime'] = pd.to_datetime(data['pickup_datetime'], errors='coerce')
data['pickup_hour'] = data['pickup_datetime'].dt.hour
data['pickup_day'] = data['pickup_datetime'].dt.day
data['pickup_month'] = data['pickup_datetime'].dt.month
data['pickup_dayofweek'] = data['pickup_datetime'].dt.dayofweek

# 
# Calculate distance using geopy
data['distance_miles'] = [
    round(distance((data.pickup_latitude[i], data.pickup_longitude[i]),
                   (data.dropoff_latitude[i], data.dropoff_longitude[i])).miles, 2)
    for i in data.index
]

# Prepare the data for regression
# Drop columns we no longer need
data = data.drop(columns=['pickup_datetime', 'pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude'])


# # Filter based on suggested thresholds
data = data[(data['fare_amount'] > 0) & (data['fare_amount'] <= 113)]
data = data[(data['distance_miles'] > 0) & (data['distance_miles'] <= 45)]


# Define features and target variable
X = data[['distance_miles', 'pickup_hour', 'pickup_day', 'pickup_month', 'pickup_dayofweek', 'passenger_count']]
y = data['fare_amount']

# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize the model
model = LinearRegression()

# Train the model on the training data
model.fit(X_train, y_train)

# Make predictions on the test data
y_pred = model.predict(X_test)

# Evaluate the model
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("Mean Absolute Error (MAE):", mae)
print("Root Mean Squared Error (RMSE):", rmse)
print("R-squared (R²):", r2)


# Print the model coefficients and intercept
print("Model Coefficients:", model.coef_)
print("Intercept:", model.intercept_)


Mean Absolute Error (MAE): 2.2582283259891898
Root Mean Squared Error (RMSE): 4.172989097542776
R-squared (R²): 0.8020824649867492
Model Coefficients: [ 3.74925282e+00  1.14281490e-02  5.14826049e-05  3.40275550e-02
 -4.75041843e-02  5.09201047e-02]
Intercept: 3.1618189012618565
